In [20]:
import os
import pandas as pd
import yaml
import random
from faker import Faker
fake = Faker('pt_BR')

def load_config(file_path: str):
    """Carrega arquivo YAML"""
    with open(file_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [39]:
# Configura o diretório de trabalho para a raiz do projeto
import os
import sys

# Define o caminho da raiz do projeto diretamente
project_root = '/home/nicolas/Projects/Personal/corporate-benefits-transactions'
os.chdir(project_root)
print(f"Diretório de trabalho atual: {os.getcwd()}")

# Adiciona o diretório raiz ao path do Python para imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Carrega configurações usando caminhos relativos (agora que o diretório está correto)
config_files_updated = load_config('./config/config_files_updated.yaml')
print("Configurações carregadas com sucesso!")

Diretório de trabalho atual: /home/nicolas/Projects/Personal/corporate-benefits-transactions
Configurações carregadas com sucesso!


### 1. Municipalities

In [22]:
# Recarregando o módulo para garantir que temos a versão corrigida
import importlib
import src.extractors.municipality_data_importer
importlib.reload(src.extractors.municipality_data_importer)
from src.extractors.municipality_data_importer import generate_municipality_data

shapefile_path = './data/raw/municipios/municipios.shp'

### 2. Products

In [28]:
def generate_products(config_files_updated):
    """Generates products DataFrame based on business rules"""
    produtos = []
    for codigo, dados in config_files_updated['products'].items():
        produto = {
            'product_id': f"PROD_{dados['code'].upper()}",
            'code': dados['code'],
            'name': dados['name'],
            'category': dados['category'],
            'description': f"Produto de {dados['name']}",
            'active': True,
            'created_at': pd.Timestamp.now()
        }
        produtos.append(produto)

    produtos_df = pd.DataFrame(produtos)
    produtos_df['code'] = produtos_df['code'].astype('category')
    produtos_df['name'] = produtos_df['name'].astype('category')
    produtos_df['category'] = produtos_df['category'].astype('category')

    return produtos_df

### 3. CNAE

In [ ]:
from src.extractors.kaggle_downloader import download_kaggle_dataset

def extract_cnae_data():
    download_kaggle_dataset("diovanimerlo/cnae-brazil-2024")

    cnaes = pd.read_csv(
        './data/raw/cnae_dataset_2024.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "secao_id": str,
            "secao": "category",
            "divisao_id": str,
            "divisao": "category",
            "grupo_id": str,
            "grupo": "category",
            "classe_id": str,
            "classe": "category",
            "subclasse_id": str,
            "subclasse": "category"
        }
    )
    return cnaes

Path to dataset files: /home/nicolas/.cache/kagglehub/datasets/diovanimerlo/cnae-brazil-2024/versions/1
Arquivo já existe, pulando: /home/nicolas/Projects/Personal/corporate-benefits-transactions/src/extractors/../../data/raw/cnae_dataset_2024.csv


### 4. Segments

In [ ]:
def generate_segments(config_files_updated):
    """Generate segments DataFrame from business rules"""
    segments_list = config_files_updated['company_segments']
    segments = pd.DataFrame(segments_list)
    segments['created_at'] = pd.Timestamp.now()
    segments['name'] = segments['name'].astype('category')
    segments['category'] = segments['category'].astype('category')
    return segments


### 5. Acquirers

In [ ]:
def generate_acquirers(config_files_updated):
    """Generates DataFrame for Acquirers based on business rules"""
    adquirentes = []
    for idx, name in enumerate(config_files_updated['acquirers'], start=1):
        acquirer = {
            "acquirer_id": f"ACQ_{idx:03d}",
            "name": name,
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        adquirentes.append(acquirer)
    return pd.DataFrame(adquirentes)

### 6. Transaction Modalities

In [ ]:
def generate_transaction_modalities(config_files_updated):
    """Generates DataFrame for Transaction Modalities based on business rules"""
    modalidades = []
    modalidades_yaml = config_files_updated['transaction_modalities']

    # Se for dict, usa .values(); se for lista, itera direto
    iterator = modalidades_yaml.values() if isinstance(modalidades_yaml, dict) else modalidades_yaml

    for idx, dados in enumerate(iterator, start=1):
        modality = {
            "modality_id": f"MOD_{idx:03d}",
            "code": dados["name"].upper().replace(" ", "_"),
            "name": dados["name"],
            "type": "brand" if dados["name"].upper() in ["ELO", "MASTERCARD", "VISA"] else "proprietary_network",
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        modalidades.append(modality)
    return pd.DataFrame(modalidades)

### 7. MCC Categories

In [31]:
import pandas as pd

def extract_mcc_data():
    mcc = pd.read_csv(
        './data/raw/mcc.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "mcc_id": str,
            "mcc_code": str,
            "mcc_description": str,
            "valid_payment_brands": str,
            "category": "category"
        }
    )
    return mcc

### 8. Capture Technologies

In [ ]:
def generate_capture_technologies():
    """Generate capture technologies"""
    technologies = [
        {"code": "CHIP", "category": "physical", "name": "Chip", "requires_physical_card": True},
        {"code": "MAG_STRIPE", "category": "physical", "name": "Magnetic Stripe", "requires_physical_card": True},
        {"code": "CONTACTLESS", "category": "physical", "name": "Contactless", "requires_physical_card": True},
        {"code": "MOBILE_NFC", "category": "digital", "name": "Mobile NFC", "requires_physical_card": False},
        {"code": "APPLE_PAY", "category": "digital", "name": "Apple Pay", "requires_physical_card": False},
        {"code": "GOOGLE_PAY", "category": "digital", "name": "Google Pay", "requires_physical_card": False}
    ]

    capture_tech_list = []
    for idx, tech in enumerate(technologies, start=1):
        capture_tech = {
            "technology_id": f"TECH_{idx:03d}",
            "technology_code": tech["code"],
            "technology_category": tech["category"],
            "technology_name": tech["name"],
            "requires_physical_card": tech["requires_physical_card"],
            "created_at": pd.Timestamp.now()
        }
        capture_tech_list.append(capture_tech)

    return pd.DataFrame(capture_tech_list)

### 9. Establishments 

In [43]:
def generate_establishments(n_establishments, df_cnae, df_mcc, df_municipalities):
    """Generate establishments DataFrame based on existing CNAE, MCC and municipalities data"""
    establishments = []
    
    # Tipos de natureza jurídica mais comuns
    legal_natures = ["LTDA", "EIRELI", "SA", "MEI", "EPP", "COOPERATIVA"]
    company_sizes = ["MEI", "ME", "EPP", "PEQUENO PORTE", "MEDIO PORTE", "GRANDE PORTE"]
    statuses = ["ATIVA", "INATIVA", "SUSPENSA", "BAIXADA"]
    status_reasons = ["NORMAL", "IRREGULARIDADE", "FALTA DE MOVIMENTO", "MUDANÇA DE ENDEREÇO"]
    
    for i in range(n_establishments):
        establishment = {
            'establishment_id': f"EST_{i:06d}",
            'cnpj': fake.cnpj(),
            'legal_name': fake.company(),
            'trade_name': fake.company_suffix() + " " + fake.last_name(),
            'main_cnae_id': random.choice(df_cnae['subclasse_id'].tolist()),
            'mcc_code': random.choice(df_mcc['mcc_code'].tolist()),
            'legal_nature': random.choice(legal_natures),
            'company_size': random.choice(company_sizes),
            'municipality_id': random.choice(df_municipalities['municipality_id'].tolist()),
            'postal_code': fake.postcode(),
            'street': fake.street_name(),
            'number': str(fake.building_number()),
            'complement': random.choice([None, "Sala " + str(random.randint(1, 100)), "Loja " + str(random.randint(1, 50))]),
            'neighborhood': fake.neighborhood(),
            'email': fake.company_email(),
            'phone': fake.phone_number(),
            'registration_status': random.choice(statuses),
            'status_date': fake.date_between(start_date='-2y', end_date='today'),
            'status_reason': random.choice(status_reasons),
            'opening_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'created_at': pd.Timestamp.now()
        }
        establishments.append(establishment)
    
    return pd.DataFrame(establishments)

### 10. Companies

In [48]:
def generate_companies(n_companies, municipalities, segments):
    companies = []
    for i in range(n_companies):
        company = {
            'company_id': f"EMP_{i}",
            'parent_company_id': random.choice([None] + [c["company_id"] for c in companies]) if companies else None,
            'name': fake.company(),
            'cnpj': fake.cnpj(),
            'company_type': random.choice(["Privada", "Publica"]),
            'segment_id': random.choice(segments['segment_id'].tolist()),
            'municipality_id': random.choice(municipalities['municipality_id'].tolist()),
            'contract_registration_date': fake.date_between(start_date='-5y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        companies.append(company)
    return pd.DataFrame(companies)  # Return as DataFrame for consistency

### 11. Cardholders

In [50]:
def generate_cardholders(n_funcionarios, companies_df, municipalities_df):
    """Generate cardholders DataFrame based on existing companies data"""
    cardholders = []
    
    for i in range(n_funcionarios):
        cardholder = {
            'cardholder_id': f"FUNC_{i}",
            'company_id': random.choice(companies_df['company_id'].tolist()),
            'name': fake.name(),
            'cpf': fake.cpf(),
            'municipality_id': random.choice(municipalities_df['municipality_id'].tolist()),
            'admission_date': fake.date_this_decade(),
            'position': fake.job(),
            'department': fake.word(),
            'status': random.choice(['active', 'inactive', 'terminated']),
            'created_at': pd.Timestamp.now()
        }
        cardholders.append(cardholder)
    
    return pd.DataFrame(cardholders)

### 12. Cards

In [52]:
def generate_cards(n_cards, cardholders_df, modality_df):
    cards = []

    for idx in range(n_cards):
        card_number = "".join([str(random.randint(0, 9)) for _ in range(16)])
        formatted_card_number = " ".join([card_number[i:i+4] for i in range(0, 16, 4)])
        
        cvv = "".join([str(random.randint(0, 9)) for _ in range(3)])

        card = {
            "card_id": idx,
            "cardholder_id": random.choice(cardholders_df['cardholder_id'].tolist()),
            "card_number": formatted_card_number,
            "modality_id": random.choice(modality_df['modality_id'].tolist()),
            "expiration_date": fake.date_between(start_date='today', end_date='+5y'),
            "cvv": cvv,
            "card_type": random.choice(['credit', 'debit']),
            "issuance_date": pd.Timestamp.now(),
            "status": random.choice(['active', 'inactive']),
            "created_at": pd.Timestamp.now()
        }
        cards.append(card)

    return pd.DataFrame(cards)

### 13. Card Products (TODO)

In [53]:
def generate_card_products(cards_df, products_df, min_products=1, max_products=3):
    """
    Gera produtos associados aos cartões.
    Cada cartão pode ter entre min_products e max_products produtos.
    """
    card_products = []

    for card_id in cards_df['card_id'].tolist():
        n_products = random.randint(min_products, max_products)

        for _ in range(n_products):
            activation_date = fake.date_between(start_date='-2y', end_date='today')

            # 30% de chance de já ter sido desativado
            if random.random() < 0.3:
                deactivation_date = fake.date_between(start_date=activation_date, end_date='today')
                status = random.choice(['suspended', 'cancelled'])
            else:
                deactivation_date = None
                status = 'active'

            card_product = {
                "card_product_id": fake.uuid4(),
                "card_id": card_id,
                "product_id": random.choice(products_df['product_id'].tolist()),
                "current_balance": round(random.uniform(0, 5000), 2),
                "daily_limit": round(random.uniform(100, 2000), 2),
                "monthly_limit": round(random.uniform(1000, 10000), 2),
                "activation_date": activation_date,
                "deactivation_date": deactivation_date,
                "status": status,
                "created_at": pd.Timestamp.now()
            }
            card_products.append(card_product)

    return pd.DataFrame(card_products)


### 14.Terminals

In [ ]:
def generate_terminals(n_terminals, establishments_df, acquirers_df):
    """Generate terminals for establishments"""
    terminals = []

    terminal_models = ["POS Cielo LIO", "TEF Rede", "Stone Ton", "GetNet POS", "Moderninha Pro"]
    terminal_types = ["POS", "PDV", "TEF"]

    for i in range(n_terminals):
        terminal = {
            'terminal_id': f"TERM_{i:08d}",
            'terminal_code': f"T{random.randint(100000, 999999)}",
            'establishment_id': random.choice(establishments_df['establishment_id'].tolist()),
            'acquirer_id': random.choice(acquirers_df['acquirer_id'].tolist()),
            'terminal_model': random.choice(terminal_models),
            'terminal_type': random.choice(terminal_types),
            'status': random.choice(['active', 'inactive', 'maintenance']),
            'installation_date': fake.date_between(start_date='-3y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        terminals.append(terminal)

    return pd.DataFrame(terminals)

### Generate Response Codes

In [ ]:
def generate_response_codes():
    """Generate response codes for transaction processing"""
    response_codes = [
        {"code": "00", "description": "Approved", "category": "success", "is_fraud": False},
        {"code": "51", "description": "Insufficient funds", "category": "balance", "is_fraud": False},
        {"code": "14", "description": "Invalid card number", "category": "card", "is_fraud": True},
        {"code": "55", "description": "Incorrect PIN", "category": "password", "is_fraud": False},
        {"code": "96", "description": "System error", "category": "network", "is_fraud": False},
        {"code": "30", "description": "Format error", "category": "network", "is_fraud": False},
        {"code": "12", "description": "Invalid transaction", "category": "establishment", "is_fraud": True},
        {"code": "43", "description": "Stolen card", "category": "card", "is_fraud": True}
    ]

    codes_list = []
    for idx, code_info in enumerate(response_codes, start=1):
        response_code = {
            "response_code_id": f"RESP_{idx:03d}",
            "response_code": code_info["code"],
            "response_description": code_info["description"],
            "response_category": code_info["category"],
            "is_potential_fraud": code_info["is_fraud"],
            "created_at": pd.Timestamp.now()
        }
        codes_list.append(response_code)

    return pd.DataFrame(codes_list)


###  DataFrames

In [55]:
municipalities_df = generate_municipality_data(shapefile_path)
products_df = generate_products(config_files_updated)
cnaes_df = extract_cnae_data()
segments_df = generate_segments(config_files_updated)
acquirers_df = generate_acquirers(config_files_updated)
transaction_modalities_df = generate_transaction_modalities(config_files_updated)
mcc_df = extract_mcc_data()
capture_tech_df = generate_capture_technologies()
establishments_df = generate_establishments(config_files_updated['data_volumes']['establishments'], cnaes_df, mcc_df, municipalities_df)
companies_df = generate_companies(config_files_updated['data_volumes']['companies'], municipalities_df, segments_df)
cardholders_df = generate_cardholders(config_files_updated['data_volumes']['cardholders'], companies_df, municipalities_df)
cards_df = generate_cards(config_files_updated['data_volumes']['cards'], cardholders_df, transaction_modalities)
card_products_df = generate_card_products(cards_df, products_df, min_products=1, max_products=3)

Path to dataset files: /home/nicolas/.cache/kagglehub/datasets/diovanimerlo/cnae-brazil-2024/versions/1
Arquivo já existe, pulando: /home/nicolas/Projects/Personal/corporate-benefits-transactions/src/extractors/../../data/raw/cnae_dataset_2024.csv


## OLTP process

In [ ]:
def process_cardholders_to_oltp():
    """
    Process RAW cardholders to OLTP format
    Applies security transformations (hashing sensitive data)
    """
    oltp_cardholders = cardholders.copy()
    
    # Hash do CPF para segurança no OLTP
    oltp_cardholders['cpf_hash'] = oltp_cardholders['cpf'].apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()[:32]
    )
    
    # Remove CPF original (fica apenas o hash no OLTP)
    oltp_cardholders.drop('cpf', axis=1, inplace=True)
    
    # Adiciona campo updated_at requerido pelo OLTP
    oltp_cardholders['updated_at'] = pd.Timestamp.now()
    
    return oltp_cardholders

def transaction_processor(n_transactions, cardholders):
    transactions = []

    for _ in range(n_transactions):
        transaction = {
            "transaction_id": fake.uuid4(),
            "cardholder_id": cardholders.sample().iloc[0]["cardholder_id"],
            "amount": round(random.uniform(1, 1000), 2),
            "currency": "BRL",
            "status": random.choice(["approved", "declined"]),
            "created_at": pd.Timestamp.now()
        }
        transactions.append(transaction)

    return transactions
